In [0]:
'''1. Customer Purchase analysis with PySpark.
Calculate the total purchase amount for each customer

sample data
'''

from pyspark.sql import SparkSession
#from pyspark.sql import functions as F
#or
from pyspark.sql.functions import sum
spark = SparkSession.builder.appName("CustomerPurchaseAnalysis").getOrCreate()

data = [
    (1, 100, "2023-01-15"),
    (2, 150, "2023-02-20"),
    (1, 200, "2023-03-10"),
    (3, 50, "2023-04-05"),
    (2, 120, "2023-05-15"),
    (1, 300, "2023-06-25")
]
columns = ["customer_id", "purchase_amount", "purchase_date"]

df=spark.createDataFrame(data,columns)
df.show()

#DSL
df.groupBy("customer_id").agg(sum("purchase_amount").alias("Total_amt")).show()
#total_purchase_per_customer = df.groupBy("customer_id").agg(F.sum("purchase_amount").alias("total_purchase_amount"))
#SQL
df.createOrReplaceTempView("tbl")
ds_sql=spark.sql("select customer_id,sum(purchase_amount) as Total_amt from tbl group by customer_id")
ds_sql.show()

In [0]:
'''2.FIND THE CUSTOMER WITH THE HIGHEST
 TOTAL PURHASE AMOUNT

sample data'''

data = [
    (1, 100, "2023-01-15"),
    (2, 150, "2023-02-20"),
    (1, 200, "2023-03-10"),
    (3, 50, "2023-04-05"),
    (2, 120, "2023-05-15"),
    (1, 300, "2023-06-25")
]
columns = ["customer_id", "purchase_amount", "purchase_date"]
#DSL
df=spark.createDataFrame(data,columns)
df.show()
df1=df.groupBy("customer_id").agg(sum("purchase_amount").alias("Total_amt"))
df2=df1.orderBy(df1.Total_amt.desc())
customer_with_highest_purchase=df2.first()
print(customer_with_highest_purchase)
print("customer_with_highest_purchase is" ,customer_with_highest_purchase["customer_id"] ,"amount is ",customer_with_highest_purchase["Total_amt"])

print(
    f"Customer with highest purchase: {customer_with_highest_purchase['customer_id']} \n"
    f"Customer with highest Amount:  {customer_with_highest_purchase['Total_amt']} ")




In [0]:
'''
Question 2 using SQL
#SQL
'''

df.createOrReplaceTempView("tbl")
sql("select * from tbl").show()
df_sql=sql("select customer_id,sum(purchase_amount) as total_amt from tbl group by customer_id order by total_amt desc")
top_customer=df_sql.first()
print(top_customer)
#Preferred way
print(
    f"Customer with highest purchase: {top_customer['customer_id']} "
    f"| Total Amount: {top_customer['total_amt']}"
)
#if not using f string u have to convert to string
#print(top_customer["customer_id"]) - returning int. when we append with str, python wont accept
print("Customer ID: " +  str(top_customer["customer_id"]) + "Total_amt:" + str(top_customer['total_amt'])
      )





In [0]:
'''
3.Calculate the total revenue generated from all sales

sample data
'''
from pyspark.sql.functions import sum,col,avg
columns = ["product_id", "product_name", "category", "price", "quantity_sold"]
data = [
    (1, "Product A", "Electronics", 500, 100),
    (2, "Product B", "Clothing", 50, 200),
    (3, "Product C", "Electronics", 800, 50),
    (4, "Product D", "Beauty", 30, 300),
    (5, "Product E", "Clothing", 75, 150)
]

#DSL
df=spark.createDataFrame(data,columns)
df.show()
revenue = df.withColumn("revenue",col("price")*col("quantity_sold")).agg(sum("revenue").alias("total_revenue")).first()
print(revenue['total_revenue'])
#SQL
df.createOrReplaceTempView("sales_tbl")
spark.sql("select sum(price*quantity_sold) as revenue from sales_tbl").first()
print(revenue['total_revenue'])
#OR
spark.sql("select sum(price*quantity_sold) as revenue from sales_tbl").first()['revenue']
#Ref for first

df.createOrReplaceTempView("sales_tbl")

rows = spark.sql("select product_id, price from sales_tbl").first()
print(rows)
print(rows['product_id'])
print(rows['price'])


## Question 3.1: Top 5 best-selling products
#DSL
print('DSL Way')
df.orderBy(col("quantity_sold").desc()).show()

print('SQL Way')
df.createOrReplaceTempView("sales_tbl")
sql("select * from sales_tbl order by quantity_sold desc").show()



In [0]:

## Question 3.2 - Average price per category
print('DSL Way')
df.groupBy("category").agg(avg("price").alias("avg_price")).show()
print('SQL Way')
spark.sql("select avg(price) as avg_price,category from sales_tbl group by category").show()

In [0]:
# Question 4: Category with highest total revenue

data = [
    (1, "Product A", "Electronics", 500, 100),
    (2, "Product B", "Clothing", 50, 200),
    (3, "Product C", "Electronics", 800, 50),
    (4, "Product D", "Beauty", 30, 300),
    (5, "Product E", "Clothing", 75, 150)
]
columns = ["product_id", "product_name", "category", "price", "quantity_sold"]
#DSL
df=spark.createDataFrame(data,columns)

df.createOrReplaceTempView("sales_tbl")
#SQL
sql("select * from sales_tbl").show()
sql("select   category, sum(price*quantity_sold) as total_revenue from sales_tbl group by category order by total_revenue desc").show()
sql("select   category, sum(price*quantity_sold) as total_revenue from sales_tbl group by category order by total_revenue desc")

#DSL
df.show()
from pyspark.sql.functions import *
revenue_category=df.withColumn("total_revenue",col("price")*col("quantity_sold")).groupBy("category").agg(sum("total_revenue").alias("total_revenue"))
revenue_category.show()
max_revenue_category=revenue_category.orderBy(col("total_revenue").desc()).first()
print(f"category with highest revenue is {max_revenue_category['total_revenue']}")
 
 

In [0]:
#Qn 4:
'''
Dataset: The dataset is in CSV format and 
contains the following columns:
 employee_id, employee_name, department, salary.

Questions:

Calculate the total payroll cost for the company.

Find the average salary for each department.

Identify the highest-paid employee and their department.

Calculate the total number of employees in each department.

Sample Dataset:


'''

data = [
    (1, "John Doe", "Engineering", 90000),
    (2, "Jane Smith", "Marketing", 75000),
    (3, "Michael Johnson", "Engineering", 105000),
    (4, "Emily Davis", "Marketing", 80000),
    (5, "Robert Brown", "Engineering", 95000),
    (6, "Linda Wilson", "HR", 60000)
]
columns = ["employee_id", "employee_name", "department", "salary"]

emp_df=spark.createDataFrame(data,columns)
emp_df.show()

#Total payrol cost 
from pyspark.sql.functions import *
total_cost=emp_df.agg(sum("salary").alias("Total_cost"))
total_cost.show()
#AVG Sal for each dept
from pyspark.sql.functions import *
avg_sal_dept_wise_df=emp_df.groupBy("department").agg(avg("salary").alias("avg_cost"))
avg_sal_dept_wise_df.show()

#Identify the highest-paid employee and their department.
highest_paid_emp=emp_df.orderBy(col("salary").desc()).first()
print(highest_paid_emp)
print(f"employee is {highest_paid_emp['employee_name']} and highest paid salary is {highest_paid_emp['salary']}")
#Calculate the total number of employees in each department.
count_emp=emp_df.groupBy("department").count()
#OR
count_emp=emp_df.groupBy("department").agg(count("*").alias("emp_count"))
count_emp.show()
 
 

In [0]:
#Qn 5
#sample dataset
#Calculate the total number of orders for each customer.
#
data = [
    (1, "C101", "2023-07-01", 150),
    (2, "C102", "2023-07-02", 200),
    (3, "C101", "2023-07-02", 100),
    (4, "C103", "2023-07-03", 300),
    (5, "C102", "2023-07-04", 250),
    (6, "C101", "2023-07-05", 120)
]
columns = ["order_id", "customer_id", "order_date", "total_amount"]
order_df=spark.createDataFrame(data,columns)
order_df.show()
#DSL
from pyspark.sql.functions import *
total_orders_per_customer=order_df.groupBy("customer_id").agg(count("*"))
total_orders_per_customer.show()

In [0]:
#Qn 6 
'''

1: Average score per subject

2: Highest score and corresponding student per subject

3: Total number of students per subject

4: Subject(s) with the highest average score

sample dataset

'''
data = [
    (1, "Math", 85),
    (2, "Science", 92),
    (3, "Math", 78),
    (4, "English", 88),
    (5, "Science", 95),
    (6, "Math", 90)
]
columns = ["student_id", "subject", "score"]

student_df=spark.createDataFrame(data,columns)
student_df.show()
from pyspark.sql.functions import *
#Average score per subject
student_df1=student_df.groupBy("subject").agg(avg("score").alias("avg_score"))
student_df1.orderBy(col("avg_score").desc()).show()


#Highest score and corresponding student per subject
#using DSL
highest_score_per_subject = student_df.groupBy("subject").agg(max("score").alias("highest_score"))
highest_score_per_subject.show()
highest_score_students=student_df.join(highest_score_per_subject,on="subject",how="inner").filter(col("score")==col("highest_score"))
highest_score_students.show()

student_df.show()
student_df.createOrReplaceTempView("student")
high_score_df = spark.sql("select subject,max(score) as highest_score from student group by subject ")
high_score_df.show()
high_score_df.createOrReplaceTempView("high_score")
sql("select s.subject,s.student_id,s.score,h.highest_score from student s join high_score h where s.subject=h.subject and s.score=h.highest_score").show()
 


In [0]:
# Question 7

'''
7.Calculate the total revenue generated from all orders
	1: Total revenue generated from all orders
	2: Top 5 orders with highest total amount

sample data
'''


order_data = [
    (1, "C101", "2023-07-01", 150),
    (2, "C102", "2023-07-02", 200),
    (3, "C101", "2023-07-02", 100),
    (4, "C103", "2023-07-03", 300),
    (5, "C102", "2023-07-04", 250),
    (6, "C101", "2023-07-05", 120)
]
order_columns = ["order_id", "customer_id", "order_date", "total_amount"]

product_data = [
    (1, "Product A", 500),
    (2, "Product B", 50),
    (3, "Product C", 800),
    (4, "Product D", 30),
    (5, "Product E", 75)
]
product_columns = ["product_id", "product_name", "price"]

order_df=spark.createDataFrame(order_data,order_columns)
product_df=spark.createDataFrame(product_data,product_columns)
order_df.show()
product_df.show()

#Total revenue generated from all orders
order_df.selectExpr("sum(total_amount) as total_revenue").show()
#or
order_df.agg(sum("total_amount")).show()
row=order_df.selectExpr("sum(total_amount) as total_revenue").first()
print(row.total_revenue)



In [0]:
#Question 8
'''F𝐢𝐧𝐝 𝐭𝐡𝐞 𝐞𝐚𝐫𝐥𝐢𝐞𝐬𝐭 𝐚𝐧𝐝 𝐥𝐚𝐭𝐞𝐬𝐭 𝐭𝐢𝐦𝐞𝐬𝐭𝐚𝐦𝐩𝐬 𝐢𝐧 𝐭𝐡𝐞 𝐝𝐚𝐭𝐚𝐬𝐞𝐭
'''

from pyspark.sql.functions import *
columns = ["user_id", "timestamp"]
data = [
    ("user1", "2023-08-21 10:00:00"),
    ("user2", "2023-08-21 11:30:00"),
    ("user1", "2023-08-21 12:15:00"),
    ("user3", "2023-08-21 13:45:00"),
    ("user2", "2023-08-21 14:30:00"),
    ("user1", "2023-08-21 15:00:00")
]

df=spark.createDataFrame(data,columns)
df.show()
# 1. Find earliest and latest timestamps
df.printSchema()
new_df=df.withColumn("timestamp",col("timestamp").cast("timestamp"))
new_df.show()
#SQL
new_df.createOrReplaceTempView("temp")
spark.sql("select min(timestamp) as earliest,max(timestamp) as latest from temp").show()
#DSL
new_df.agg(min("timestamp").alias("earliest"),max("timestamp").alias("latest")).show()


# 2. Count the number of activities per user
activity_count = new_df.groupBy("user_id").agg(count("timestamp").alias("activity_count"))
activity_count.show()

#3.Calculate time duration between consecutive activities for each user
query1="""
select user_id,time1, previous_date,unix_timestamp(time1)-unix_timestamp(previous_date) as diff from (
select user_id,timestamp as time1,lag(timestamp) over(partition by user_id order by timestamp) previous_date  from temp t
)
"""
sql(query1).show()

#using DSL
window_spec = Window.partitionBy("user_id").orderBy("timestamp")
df2 = df2.withColumn("prev_timestamp", lag("timestamp").over(window_spec))
df2 = df2.withColumn("time_diff", (unix_timestamp("timestamp") - unix_timestamp("prev_timestamp")).cast("int"))
df2.show()





In [0]:
# Qn. count the action
data = [
    (1, "login", "2023-08-20 10:23:45"),
    (2, "view", "2023-08-20 11:15:30"),
    (1, "purchase", "2023-08-20 12:45:18"),
    (3, "view", "2023-08-20 13:30:22")
]
columns = ["user_id", "action", "timestamp"]
df=spark.createDataFrame(data,columns)
df.show()

# Convert timestamp to timestamp type
df2 = df.withColumn("timestamp", col("timestamp").cast("timestamp"))
df2.count()
#Uniq actions in the dataset

df2.select (col("action")).distinct().show()

In [0]:
'''
11.Question 1: Calculate Average User Session Duration
You have a dataset containing user activity logs in a 
PySpark DataFrame with the following columns:
 user_id, timestamp, and action. 
The action column indicates whether the user
 started or ended a session. It can have values 
'start' or 'end'. Your task is to calculate the 
average duration of user sessions.

 Sample Data

'''

from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag, unix_timestamp, avg, when

# Initialize Spark Session
spark = SparkSession.builder.appName("AverageSessionDuration").getOrCreate()

# Sample Data
data = [
    (1, "2022-01-01 10:00", "start"),
    (1, "2022-01-01 10:15", "end"),
    (2, "2022-01-01 11:00", "start"),
    (1, "2022-01-01 11:30", "start"),
    (2, "2022-01-01 11:45", "end"),
    (1, "2022-01-01 12:00", "end"),
]
columns = ["user_id", "timestamp", "action"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Convert timestamp to proper TimestampType
df = df.withColumn("timestamp", col("timestamp").cast("timestamp"))
df.show()

# Define window partitioned by user_id and ordered by timestamp
window_spec= Window.partitionBy("user_id").orderBy("timestamp")

# Pair start and end actions: Use lag() to find the previous action and timestamp
df_with_lag=df.withColumn("previous_action",lag("action").over(window_spec)) \
    .withColumn("previous_timestamp",lag("timestamp").over(window_spec))
df_with_lag.show()   

# Filter for valid session pairs (start -> end)
valid_sessions = df_with_lag.filter((col("action") == "end") & (col("previous_action") == "start"))
valid_sessions.show()


In [0]:
'''15.What are the different ways to handle row 
duplication in a PySpark DataFrame? '''

from pyspark.sql import SparkSession

# Create a Spark session
spark = SparkSession.builder.appName("RemoveDuplicates").getOrCreate()

data = [("James", "Sales", 3000),
        ("Michael", "Sales", 4600),
        ("Robert", "Sales", 4100),
        ("Maria", "Finance", 3000),
        ("James", "Sales", 3000),
        ("Scott", "Finance", 3300),
        ("Jen", "Finance", 3900),
        ("Jeff", "Marketing", 3000),
        ("Kumar", "Marketing", 2000),
        ("Saif", "Sales", 4100)]
columns = ["Name", "Department", "Salary"]
# Create DataFrame
df = spark.createDataFrame(data, schema=columns)
df.show()
df_without_dup=df.distinct()
df_without_dup.show()

#Or using dropduplicates
df_drop_dup=df.dropDuplicates(["name"])
df_drop_dup.show()




In [0]:
#using SQL
data = [("James", "Sales", 100),
        ("James", "Sales", 30000),
        ("Scott", "Finance", 100),
        ("Scott", "Finance", 3900),
        ("Jeff", "Marketing", 100),
        ("Jeff", "Marketing", 20000)]
columns = ["Name", "Department", "Salary"]
# Create DataFrame
df = spark.createDataFrame(data, schema=columns)
df.show()

#Remove duplicates based on Name and Department
df.createOrReplaceTempView("temp")
spark.sql("select * from (select row_number() over(partition by   Name,Department order by Salary desc) as rn,* from  temp) where rn=1").show()


# Same using DSL
from pyspark.sql.functions import row_number,rank,dense_rank
from pyspark.sql.window import Window
w_spaec = Window.partitionBy("Department").orderBy(col("Salary").desc())
df3=df.withColumn("row_num",row_number().over(w_spaec)).filter(col("row_num")==1).drop("row_num")
df3.show()

In [0]:
#Nth Max Salary
'''
SELECT Salary
FROM (
    SELECT Salary,
           ROW_NUMBER() OVER (ORDER BY Salary DESC) AS rn
    FROM temp
) t
WHERE rn = N;   -- Replace N with the nth position you want
'''

In [0]:
#Qn 16

#Using DSL
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode

data = [(2,1, ["apple", "apple", "apple"]), (3,2, ["grape", "orange","apple","banana"])]
df = spark.createDataFrame(data, ["rer","id", "fruits"])
df.select("rer","id",explode("fruits").alias("fruit")).distinct().show()

from pyspark.sql.functions import explode


#USing SQL


# Create a Spark session
spark = SparkSession.builder.appName("explode_example").getOrCreate()

# Create a DataFrame with an array column
data = [(2,1, ["apple", "apple", "apple"]), (3,2, ["grape", "orange","apple","banana"])]
df = spark.createDataFrame(data, ["rer","id", "fruits"])
df.show(15,truncate=False)

df.createOrReplaceTempView("fruits")
spark.sql("select distinct rer,id,explode(fruits) as fruit1 from fruits ").show()



In [0]:
#Qn 17

from pyspark.sql import SparkSession
from pyspark.sql.functions import explode

# Create a Spark session
spark = SparkSession.builder.appName("explode_example").getOrCreate()
Data=[("India","1",["Kerala","Andhra Pradesh","Karnataka","Tamil Nadu","Telangana","Assam" ,"Bihar","Punjab","Karnataka"]),("canada","2",["Ontario","Alberta"])]
Shema=["Country","id", "States"]

df = spark.createDataFrame(Data, ["Country","id", "States"])
df.show(truncate=False)
# Use explode to transform the array column into separate rows
exploded_df = df.select("Country","id", explode("States").alias("States"))

exploded_df.show(truncate=False)

In [0]:
#sample data
data = [
    ("2021-01-01", "USA", 10000),
    ("2021-01-01", "India", 8000),
    ("2021-01-02", "USA", 10500),
    ("2021-01-02", "India", 8200),
    # Add more data points
]

columns = ["date", "country", "cases"]
covid_df = spark.createDataFrame(data, schema=columns)
covid_df.createOrReplaceTempView("covid_data")
spark.sql("select * from covid_data").show()

spark.sql("select date,country,cases,lag(cases) over(partition by country order by date) as Previous_cases from covid_data").show()

#Same using DSL

# Calculate total cases per country
from pyspark.sql.functions import sum
total_cases_per_country =covid_df.groupBy("country").agg(sum("cases").alias("total_cases"))
total_cases_per_country.show()

## Calculate daily new cases

from pyspark.sql.window import Window
from pyspark.sql.functions import col,lag
w_spec=Window.partitionBy("country").orderBy("date")

total_cases_per_day_dsl=covid_df.withColumn("previous_cases",lag("cases").over(w_spec))
total_cases_per_day_dsl.show()

In [0]:
#Write a PySpark program to find the user who has logged in most frequently from a dataset containing login timestamps and user IDs. Use the provided data to create a DataFrame and identify the user with the highest login count.
# Sample log data
log_data = [
    ("2023-09-11 12:00:00", "1"),
    ("2023-09-11 13:30:00", "2"),
    ("2023-09-11 14:45:00", "1"),
    ("2023-09-11 16:15:00", "3")
]
columns = ["timestamp", "user_id"]
log_df = spark.createDataFrame(log_data, columns)
log_df.show()
log_df.printSchema()

from pyspark.sql.functions import col
final_df=log_df.groupBy("user_id").count().orderBy(col("count").desc())
most_logged_user=final_df.filter("count>1")
most_logged_user_final=most_logged_user.first()
print(most_logged_user_final["user_id"])

In [0]:
'''19.Retrieve the top 2 most recent orders for each 
customer. 

Sample data'''
data = [
    (1, 101, "2023-01-15"),
    (1, 101, "2023-07-15"),
    (1, 101, "2023-07-16"),
    (2, 102, "2023-02-20"),
    (1, 103, "2023-03-10"),
    (3, 104, "2023-04-05"),
    (2, 105, "2023-05-12"),
    (2, 105, "2023-05-12")
]
schema = ["CUSTOMERID", "ORDERID", "ORDERDATE"]
order_df = spark.createDataFrame(data, schema)
order_df.createOrReplaceTempView("orders")
SQuery="""SELECT CUSTOMERID, ORDERID, ORDERDATE
FROM (
    SELECT CUSTOMERID,
           ORDERID,
           ORDERDATE,
           ROW_NUMBER() OVER (PARTITION BY CUSTOMERID ORDER BY TO_DATE(ORDERDATE) DESC) AS rn
    FROM orders
) t
WHERE rn <= 2
ORDER BY CUSTOMERID, rn"""
spark.sql("select * from orders").show()
spark.sql(SQuery).show()


#using DSL
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
# Convert the ORDERDATE column to a date type
date_df=order_df.withColumn("orderdate",col("orderdate").cast("date"))
date_df.printSchema()

# Define a window specification partitioned by CUSTOMERID and ordered by ORDERDATE in descending order
window_spec = Window.partitionBy("CUSTOMERID").orderBy(date_df["ORDERDATE"].desc())

# Add a rank column based on ORDERDATE within each customer's partition
df1 = date_df.withColumn("rank", row_number().over(window_spec))

# Filter the results to include only the top 3 most recent orders for each customer
top3_recent_orders = df1.filter(df["rank"] <= 3)

# Show the result
top3_recent_orders.show()
 

In [0]:
'''
20.Write a PySpark program with sample data and 
schema to calculate the number of visitors to a website.

Sample data



'''
from pyspark.sql.types import StructType, StructField, StringType
schema = StructType([
    StructField("Name", StringType(), True),
    StructField("VisitedWebsite", StringType(), True)])

data = [
    ("Arwen", "Yes"),
    ("Bilbo", "Yes"),
    ("Nick", "No"),
    ("Frodo", "Yes"),
    ("Merry", "No"),
    ("Luca", "Yes")
]

web_df=spark.createDataFrame(data,schema)
web_df.show()

web_df.createOrReplaceTempView("web")
visitors_count = sql("select * from web where VisitedWebsite='Yes'").count()
print(f"Number of visitors to the website: {visitors_count}")

#using DSL

# Calculate the total number of visitors who visited the website
visitors_count = web_df.filter(web_df.VisitedWebsite == "Yes").count()
print(f"Number of visitors to the website: {visitors_count}")

In [0]:
'''21.Write a PySpark program to calculate the total sales 
per day using a hard-coded dataset and schema:

sample data
'''
from pyspark.sql.types import StructType, StructField, StringType,IntegerType
schema = StructType([
    StructField("Date", StringType(), True),
    StructField("Sales", IntegerType(), True)
])

data = [
    ("2022-01-01", 100),
    ("2022-01-01", 150),
    ("2022-01-02", 200),
    ("2022-01-02", 120),
    ("2022-01-03", 80)
]

sales_df=spark.createDataFrame(data,schema)
sales_df.show()

In [0]:
'''
22.Find all customers who have never ordered anything
'''

# Sample data and schema for customers
data_customer = [(1, 'Joey'), (2, 'Ross'), (3, 'Monica'), (4, 'Phoebe')]
schema_customer = "Customer_ID int, Customer_Name string"

cust_df=spark.createDataFrame(data_customer,schema_customer)

# Sample data and schema for orders
data_order = [(1, 4), (3, 2)]
schema_order = "Order_ID int, Customer_ID int"
orders_df=spark.createDataFrame(data_order,schema_order)

cust_df.show()
orders_df.show()

join_df=cust_df.alias("a").join(orders_df.alias("d"),"customer_id","left_anti")
join_df.show()


In [0]:
#How can you filter a PySpark DataFrame to select rows where the age is greater than 30 using both the filter method and the where function?

from pyspark.sql import SparkSession

# Initialize a Spark session
spark = SparkSession.builder.appName("FilterExample").getOrCreate()

# Sample data
data = [("Alice", 25),
        ("Bob", 30),
        ("Charlie", 35),
        ("David", 40),
        ("Eve", 45)]
columns = ["Name", "Age"]
df = spark.createDataFrame(data, columns)
df.show()

df.filter("Age > 30").show()
df.where("Age > 30").show() #or filtered_df1=df.where(df.Age > 30)

In [0]:
%sql
Use to_date → when converting a string into a real date type.

Use date_format → when formatting a date/timestamp into a string with a specific pattern.

In [0]:
#How can you clean and transform a PySpark DataFrame containing sales data to standardize product names and aggregate sales per month using transformations like trim(), lower(), date_format(), and count()?

from pyspark.sql import SparkSession
from pyspark.sql.functions import trim, lower, date_format, col,count

# Create a SparkSession
spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

# Sample Sales data
data = [(1, "Toycar1", "2000-01-16"),
        (2, "toYcar2", "2000-01-17"),
        (3, "toycaR3", "2000-02-18"),
        (4, "doll1", "2000-02-19"),
        (5, "doll2", "2000-02-28"),
        (6, "data", "2000-03-31"),
        (7, "doll1", "2000-02-19")]

# Create a DataFrame
sales_df = spark.createDataFrame(data, ["sale_id", "product_name", "sale_date"])
sales_df.show()
sales_df.printSchema()

sales_df.createOrReplaceTempView("sales_tbl")
sales_df1=spark.sql("select * from sales_tbl")
sales_df1.show() 
query="""SELECT product_name, sale_dt, COUNT(sale_id) AS cnt
FROM (
    SELECT lower(product_name) AS product_name,
           to_date(sale_date,'yyyy-MM-dd') AS sale_dt,
           sale_id
    FROM sales_tbl
) t
GROUP BY product_name, sale_dt
ORDER BY sale_dt;
"""
spark.sql(query).show()

#DSL

sales_df.show()
lower_sal_df=sales_df.withColumn("product_name",lower(col("product_name"))) \
                     .withColumn("sale_date",col("sale_date").cast("date"))
lower_sal_df.show()
g_df=lower_sal_df.groupBy("product_name","sale_date") \
                .agg(count("sale_id").alias("cnt")) \
                .orderBy(col("sale_date").desc())
g_df.show()
 

In [0]:
#Get the employees department id with maximum and minimum salary in each department ?
#To find the department IDs with the maximum and minimum salary in each department,we can use the groupBy and agg functions along with max and min aggregations in PySpark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, max, min

# Create a SparkSession
spark = SparkSession.builder.appName("MaxMinSalaryPerDept").getOrCreate()

# Sample data
data = [('Lim' , 2 , 75000),
        ('Alex' , 2 , 84000),
        ('Kevin' , 2 , 80000),
        ('Theo' , 2 , 70000),
        ('Becky' , 4 , 70000),
        ('Wendy' , 4 , 85000),
        ('Maria' , 4 , 55000),
        ('Goldy' , 4 , 55000),
        ('Legolas', 5, 60000),
        ('Mike' , 5 , 65000)]

schema = "emp_name string, dept_id int, salary int"
emp_df = spark.createDataFrame(data, schema)

emp_df.createOrReplaceTempView("emp")
Query1="select dept_id,min(salary),max(salary) from emp group by dept_id"
sql(Query1).show()

#Using DSL

DSL_emp_df=emp_df.groupBy("dept_id") \
                 .agg(min("salary").alias("min_salary"),max("salary").alias("max_salary"))
DSL_emp_df.show()



df.show()

In [0]:
#How can you find the number of unique items and their total weight for each item in the given dataset using PySpark?

from pyspark.sql import SparkSession
from pyspark.sql.functions import count, sum, col

# Create a SparkSession
spark = SparkSession.builder.appName("ItemAnalysis").getOrCreate()

# Sample data
data = [
    ("JIM", "tomato", 2),
    ("SAM", "𝚊𝚙𝚙𝚕𝚎" , 2),
    ("DEAN", "𝚋𝚊𝚗𝚊𝚗𝚊" , 2),
    ("FLYNN", "tomato", 3),
    ("MIKE", "𝚝𝚊𝚌𝚘", 2),
    ("BILBO", "𝚊𝚙𝚙𝚕𝚎", 2)
]

schema = "name string, item string, weight int"
df = spark.createDataFrame(data, schema)
display(df)

result_df = df.groupBy("item").agg(
    count("item").alias("count"),
    sum("weight").alias("total_weight")
)
display(result_df)


In [0]:
#Write a PySpark query to find and report the movies with an odd-numbered movie ID and a description that is not "boring". The result should be returned in descending order by rating.

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc

# Initialize a Spark session
spark = SparkSession.builder.appName("FilterMovies").getOrCreate()

# Sample data
data = [
    (1, 'War','Great 3D', 8.9),
    (2, 'Science','Fiction', 8.5),
    (3, 'Irish','Boring', 6.2),
    (4, 'Ice song','Fantacy', 8.6),
    (5, 'House card','Interesting', 9.1)
]

schema = "ID int, Movie string, Description string, Rating double"
df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()

tran_df=df.filter((col("id") % 2 !=0) &  (col("Description") != "Boring")).orderBy(col("Rating").desc())
tran_df.show()


#SQL

df.createOrReplaceTempView("movies")

spark.sql("""SELECT * FROM movies WHERE ID % 2 != 0 AND Description != 'Boring' ORDER BY Rating DESC
""").show()


In [0]:
#If your column is already an array (like in your example ["apple", "apple", "apple"]), you don’t need split
#You can directly use explode to flatten the array into multiple rows
#If your column is a string like "apple, banana, cherry", you first need to break it into an array.

#That’s when you use split(col("fruits"), ",")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, split

# Initialize a Spark session
spark = SparkSession.builder.appName("HobbySeparator").getOrCreate()

# Sample data
data = [
    ("Ginny", "Reading, Cooking"),
    ("Mike", "Swimming, Reading"),
    ("Kevin", "Cooking, Travelling"),
    ("Amy", "Music, Painting")
]

# Define the schema
schema = ["Name", "Hobbies"]

# Create a DataFrame
df = spark.createDataFrame(data, schema)
df.show()
df.createOrReplaceTempView("hobbies")
sql("select name,explode(split(hobbies,',')) split_hobb from hobbies").show()
#using  Select Expr
df.selectExpr("Name", "explode(split(Hobbies, ',')) as split_hobb").show()

#Using DSL
df.show()

df.select("Name",explode(split(col("Hobbies"),',')).alias("hobbies")).show()




In [0]:
'''29.Write a PySpark program to process a DataFrame 
containing city names spread across three columns 
(City1, City2, City3). The program should:

1.Replace any empty strings in these columns with null
 values.
2.Create a new column that holds the greatest non-null 
 value (alphabetically) among the three city columns.
3.Display the resulting DataFrame.

SAMPLE DATA '''
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import when,col,greatest

# Initialize SparkSession
spark = SparkSession.builder.appName("CreateDataFrame").getOrCreate()

# Input data and schema
inputData2 = [('Goa', '', 'AP'), ('', 'AP', None), (None, '', 'Blr')]
schema2 = StructType([
    StructField('City1', StringType(), True),
    StructField('City2', StringType(), True),
    StructField('City3', StringType(), True)
])

# Create DataFrame
df = spark.createDataFrame(data=inputData2, schema=schema2)
df.show()

df.createOrReplaceTempView("city")
query="select greatest(nvl(City1,''),nvl(City2,'') ,nvl(City3,'')) as city,nvl(City1,'') as City1,nvl(City2,'') as City2,nvl(City3,'') as City3 from city"
sql(query).show()

#Using selectExpr
df.selectExpr("greatest(nvl(City1,''),nvl(City2,''),nvl(City3,''))as city","nvl(City1,'') as City1","nvl(City2,'') as City2","nvl(City3,'') as City3").show()

#Using DSL
df.show()
df1=df.withColumn("city1",when(col("city1")=='',None).otherwise(col("city1"))) \
      .withColumn("city2",when(col("city2")=='',None).otherwise(col("city2"))) \
      .withColumn("city3",when(col("city3")=='',None).otherwise(col("city3"))) 
df1.show()

greatest_df=df1.select(greatest(col("city1"),col("city2"),col("city3")))
greatest_df.show()

In [0]:
'''30.You are given a dataset with employee information 
containing EmployeeID, Department, Salary, and 
JoiningDate. Write a PySpark program to:

1.Calculate the average salary for each department.
2.Identify employees who earn more than the average
  salary of their respective department.'''

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

# Initialize Spark Session
spark = SparkSession.builder.appName("EmployeeAnalysis").getOrCreate()

# Input data and schema
data = [
    (1, "HR", 5000, "2022-01-15"),
    (2, "IT", 7000, "2021-06-12"),
    (3, "HR", 4500, "2023-03-20"),
    (4, "IT", 8000, "2020-12-01"),
    (5, "Finance", 6000, "2019-07-30"),
    (6, "Finance", 5500, "2020-09-25")
]
schema = ["EmployeeID", "Department", "Salary", "JoiningDate"]

# Create DataFrame
df = spark.createDataFrame(data, schema)
df.show()

# 1: Calculate the average salary for each department
avg_salary_df = df.groupBy("Department").agg(avg("Salary").alias("AvgSalary"))
avg_salary_df.show()

df.show(1)
join_df=df.select("EmployeeID","Department","Salary").join(avg_salary_df, on="Department")
df_with_avg=join_df.select("Department","EmployeeID","Salary","AvgSalary") 
df_with_avg.show(1)
above_avg_emp=df_with_avg.filter(col("Salary")>col("AvgSalary"))
above_avg_emp.select("Department","EmployeeID","Salary","AvgSalary").show() 

#SQL Way
'''
SELECT e.EmployeeID, e.Department, e.Salary, d.avg_salary
FROM employees e
JOIN (
    SELECT Department, AVG(Salary) AS avg_salary
    FROM employees
    GROUP BY Department
) d
ON e.Department = d.Department
WHERE e.Salary > d.avg_salary;


Or

SELECT *
FROM (
    SELECT 
        EmployeeID,
        Department,
        Salary,
        AVG(Salary) OVER (PARTITION BY Department) AS avg_salary
    FROM employees
) final_ds
WHERE final_ds.Salary > final_ds.avg_salary;


'''

In [0]:
rom pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, count

# Initialize Spark session
spark = SparkSession.builder.appName("TeamStats").getOrCreate()

# Sample data
data = [
    (1, "IND", "AUS", "IND"),
    (2, "IND", "NZ", "NZ"),
    (3, "AUS", "NZ", "AUS"),
    (4, "IND", "AUS", "IND"),
    (5, "NZ", "IND", "NZ")
]

# Create DataFrame
columns = ["id", "team1", "team2", "winner"]
df = spark.createDataFrame(data, columns)
df.createOrReplaceTempView("team_matches")
sql("select * from team_matches")

In [0]:
'''
34.Create a PySpark program to analyze a table 
containing null values and calculate the number of null 
and not null values for each column?

 SAMPLE DATASET


'''
from pyspark.sql.functions import col, sum, when
data = [
    ("John", 25, None),
    ("Sarah", None, "Paris"),
    (None, 30, None),
    ("Tom", 40, "London"),
    (None, None, None)
]
columns = ["name", "age", "city"]

df = spark.createDataFrame(data, columns)
df.show()


for col_name in df.columns:
    print(col_name)

expressions = []

for col_name in df.columns:
    null_count=sum(when(col(col_name).isNull(),1).otherwise(0)).alias(f"{col_name}_nulls")
    not_null_count = sum(when(col(col_name).isNotNull(), 1).otherwise(0)).alias(f"{col_name}_not_nulls")
    
    # Append both expressions for this column
    expressions.append(null_count)
    expressions.append(not_null_count)

print(expressions)
# Use the expressions in a select
df.select(expressions).show()
 

In [0]:

#using SQL

# Create temp view
df.createOrReplaceTempView("null_check")

# Get the column names from the view (same as df.columns)
columns = df.columns   # ['name', 'age', 'city']

# Build SQL dynamically
expressions = []


for col_name in columns:
    expressions.append(f"COUNT(CASE WHEN {col_name} IS NULL THEN 1 END) AS {col_name}_nulls")
    expressions.append(f"COUNT(CASE WHEN {col_name} IS NOT NULL THEN 1 END) AS {col_name}_not_nulls")

print(expressions)
sql_query = "SELECT " + ", ".join(expressions) + " FROM null_check"
print(sql_query)
# Run query
result = spark.sql(sql_query)
result.show()


In [0]:
'''
35.Write a PySpark program to find how many times a 
specific word appears in a text column of a DataFrame. 
For example, given the input text "how many times does 
the word how appear in this how string how", find how
 many times the word "how" appears.

'''

 

df=spark.createDataFrame([('word word sundar'),('word word sundar')],['text'])
df.show()

df.createOrReplaceTempView("texts")
df.show()
# Word to search
word = "word"

qry=f"""
select text,count(*) as word_count from (select 
       text,explode(split(text, ' ')) AS word
    FROM texts) tmp where lower(word) = lower('{word}')
    group by text
"""
sql(qry).show()



In [0]:
#Using DSL Way
from pyspark.sql.functions import explode,split,col

# Create a DataFrame
data = [("how many times does the word how appear in this how string how",)]
df = spark.createDataFrame(data, ["text"])

# Word to search
word = "how"

df1=df.withColumn("word",explode(split(col("text"),' ')))
df1.show()

df2=df1.filter(col("word")==f"{word}").groupBy("text").count()
df3=df2.withColumnRenamed("count","word_count")
df3.show() 

In [0]:
'''
Qn 36

#Find the number of students getting the same marks in both subjects (Maths and Science)

'''


from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

# Initialize Spark session
spark = SparkSession.builder.appName("SameMarksCount").getOrCreate()

# Sample data
data = [
    (1, 85, 85),
    (2, 75, 70),
    (3, 90, 90),
    (4, 80, 85),
    (5, 85, 85),
    (6, 95, 90),
]
columns = ["student_id", "maths", "science"]
df = spark.createDataFrame(data, columns)
df.createOrReplaceTempView("students")
sql("select maths,science,count(*) from students where maths=science group by maths,science").show()

#DSL
df_dsl = spark.createDataFrame(data, columns)
df_dsl.show()

df1=df_dsl.filter(col("maths")==col("science")).groupBy("maths","science").count().alias("number of students with same marks")
df1.show() 

In [0]:
#Qn 37 #Write a pyspark code to get the account balance for each account holder 

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, when

# Initialize Spark session
spark = SparkSession.builder.appName("AccountBalance").getOrCreate()

# Sample data
data = [
    (1, "credit", 100),
    (1, "debit", 100),
    (2, "credit", 100),
    (2, "debit", 20),
    (1, "credit", 20)
]
columns = ["account_id", "transaction_type", "amount"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

bal_df=df.groupBy("account_id").agg \
                                (sum(when(col("transaction_type")=='credit',col("amount")) \
                                    .when(col("transaction_type")=='debit',-col("amount")) \
                                    .otherwise(0)).alias("balance")
                                 )
bal_df.show()
 

In [0]:
df = spark.createDataFrame(data, columns)
df.show()
df.createOrReplaceTempView("accounts")
sql("select account_id,sum(case when transaction_type='credit' then   amount  else   -amount  end) as balance from accounts group by account_id").show()
 

In [0]:
#How to fill null values in pyspark

from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.appName("FillNullValues").getOrCreate()

# Sample data with null values
data = [
    (1, "Amy", None),
    (2, None, 25),
    (3, "Jack", None),
    (4, None, 30),
]
columns = ["id", "name", "age"]
df=spark.createDataFrame(data,columns)
df.show()

#fill null only for name

not_null_name_df=df.fillna("unk",subset=["name"])
not_null_name_df.show()

#apply null for all integer column
not_null_int_df=df.na.fill(0)
not_null_int_df.show()

#apply null for string
df_filled = df.fillna("none")  # Fills all string columns with 'none'
df_filled.show()

In [0]:
#qn 50

#Write a pyspark code to obtain a histogram of tweets posted per user in 2022. Output the tweet count per user as the bucket and the number of Twitter users who fall into that bucket.In other words, group the users by the number of tweets they posted in 2022 and count the number of users in each group.

from pyspark.sql import SparkSession
from pyspark.sql.functions import year, col, count

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("TweetHistogram") \
    .getOrCreate()

# Sample dataset representing user tweets with timestamps
data = [
    (1, "2022-01-01 10:00:00", "Tweet 1"),
    (1, "2022-02-01 14:00:00", "Tweet 2"),
    (1, "2022-03-01 18:00:00", "Tweet 3"),
    (2, "2022-04-01 09:00:00", "Tweet 1"),
    (2, "2022-05-01 12:00:00", "Tweet 2"),
    (3, "2022-06-01 15:00:00", "Tweet 1"),
    (3, "2022-07-01 16:00:00", "Tweet 2"),
    (3, "2022-08-01 17:00:00", "Tweet 3"),
    (4, "2022-09-01 19:00:00", "Tweet 1"),
    (5, "2022-10-01 20:00:00", "Tweet 1"),
    (5, "2022-11-01 21:00:00", "Tweet 2")
]

# Define schema
schema = ["UserId", "Timestamp", "Tweet"]
from pyspark.sql.functions import year, col, count
# Create DataFrame
df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()
year_df=df.withColumn("year",year(col("Timestamp")))
year_df.show()
flt_df=year_df.filter("year='2022'").groupBy("year","UserId").agg(count("Tweet").alias("Tweet_count")) 


user_tweet_buckets=flt_df.withColumn("TweetBucket", when(col("Tweet_count") <=1 , "1 bucket") \
                          .when(col("Tweet_count") <=2 , "2 bucket") \
                          .when(col("Tweet_count") <=3 , "3 bucket") \
                          .when(col("Tweet_count") <=4 , "4 bucket") \
                          .otherwise("More tweet")           
                                     )
user_tweet_buckets.show()

In [0]:
#Qn 49

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("DepartmentEmployeeCount") \
    .getOrCreate()

# Sample dataset
data = [
    ('HR', 'Male'),
    ('HR', 'Female'),
    ('IT', 'Male'),
    ('IT', 'Female'),
    ('IT', 'Male'),
    ('Sales', 'Female'),
    ('Sales', 'Male'),
    ('Sales', 'Female')
]
columns = ['department', 'gender']
df = spark.createDataFrame(data, columns)

df.createOrReplaceTempView("employees")

qry="""
SELECT department,
       COUNT(CASE WHEN gender='Male' THEN 1 END)   AS Male_count,
       COUNT(CASE WHEN gender='Female' THEN 1 END) AS Female_count,
       COUNT(*) AS dept_count
FROM employees
GROUP BY department
"""
sql(qry).show()

#using Analytical way

qry="""
select department,dept_total,count(case when gender='Male' then 1 end) as Male_count,
count(case when gender='Female' then 1 end) as Female_count
from ( 
SELECT department,
           gender,
           COUNT(*) OVER (PARTITION BY department) AS dept_total
    FROM employees
    ) tmp GROUP BY department,dept_total
 
"""
sql(qry).show()


              


In [0]:

#using DSL Way
df = spark.createDataFrame(data, columns)
df.show()

df1=df.withColumn("Male_count",when(col("gender")=="Male",1).otherwise(0)) \
    .withColumn("Female_count",when(col("gender")=="Female",1).otherwise(0))
df1.show()   
from pyspark.sql.functions import col, when, count
result_df=df1.groupBy("Department") \
            .agg(
                count("*").alias("dept_count"),
                sum("Male_count").alias("Male_count"),
                sum("Female_count").alias("Female_count")
            )
result_df.show()           

In [0]:
'''
Qn 48
from pyspark.sql import SparkSession
from pyspark.sql.functions import collect_list, concat_ws

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("CommaSeparatedSkills") \
    .getOrCreate()

# Sample data


'''

data = [
    (1, 'Joey', 'ADF'), (1, 'Joey', 'ADB'), (1, 'Joey', 'Python'),
    (2, 'Rachel', 'ADF'), (2, 'Rachel', 'SQL'), (2, 'Rachel', 'JAVA'),
    (3, 'Victor', 'ADF'), (3, 'Victor', 'SQL'), (3, 'Victor', 'DB'),
    (4, 'Mike', 'SQL'), (4, 'Mike', 'DB'), (4, 'Mike', 'SSAS'), (4, 'Mike', 'ADF')
]

# Define schema
schema = ["EmpId", "EmpName", "Skill"]

# Create DataFrame
df = spark.createDataFrame(data, schema)

df.createOrReplaceTempView("student_data")
sql("select empid,empname,concat_ws(',',collect_list(skill)) as SKills from student_data group by empid,empname").show()

 

In [0]:
#Using DSL
df = spark.createDataFrame(data, schema)
df.show(2)

df1=df.groupBy("EmpId","EmpName").agg(concat_ws(',',collect_list(col("skill"))).alias("Skills")
                                    )
df1.show()
 

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, round, when

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("StudentResultCalculation") \
    .getOrCreate()

# Sample data1 (Student ID and Name)
data1 = [
    (1, "Sam"),
    (2, "Dany"),
    (3, "Joey"),
    (4, "Lara"),
    (5, "Helen")
]

# Sample data2 (Student ID, Subject, Marks)
data2 = [
    (1, "SUB1", 90),
    (1, "SUB2", 100),
    (2, "SUB1", 70),
    (2, "SUB2", 60),
    (3, "SUB1", 30),
    (3, "SUB2", 20),
    (4, "SUB1", 50),
    (4, "SUB2", 50),
    (5, "SUB1", 45),
    (5, "SUB2", 45)
]

# Define schemas
schema1 = ["Id", "Name"]
schema2 = ["Id", "Subject", "Mark"]

# Create DataFrames
df1 = spark.createDataFrame(data1, schema=schema1)
df2 = spark.createDataFrame(data2, schema=schema2)

# Join the dataframes based on "Id"
joined_df = df1.join(df2, on="Id", how="inner")
joined_df.show()

# Calculate total marks for each student

total_df=joined_df.groupBy("Id","Name").agg(sum("Mark").alias("Total_Marks"))
total_df.show()

# Calculate percentage of total marks for each student
result_in_percentage=total_df.withColumn("Perc",((col("Total_Marks")/200)*100).cast("int"))
result_in_percentage.show()
result_in_percentage.printSchema()
result_in_percentage.withColumn("Distinction", when(col("Perc") > 90,"Distinction") \
                                                .when((col("Perc") > 80 ) &  (col("Perc") < 90),"First Class") \
                                                    .when((col("Perc") > 70 ) &  (col("Perc") < 80),"Sec Class") \
                                                .otherwise("Fail")).show()
                                
#using SelectExpr                               
result_in_percentage = result_in_percentage.selectExpr(
    "*",
    """CASE 
           WHEN Perc > 90 THEN 'Distinction'
           WHEN Perc > 80 AND Perc < 90 THEN 'First Class'
           WHEN Perc > 70 AND Perc < 80 THEN 'Sec Class'
           ELSE 'Fail'
       END AS Distinction"""
)

result_in_percentage.show()
                                
                                             